In [1]:
!pip install lightfm-next

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.0 MB/s eta 0:00:00


In [2]:
!pip list | grep lightfm

lightfm-next                             1.19.0


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, coo_matrix
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k
from tqdm.notebook import tqdm
import gc
import warnings
import seaborn as sns
import ast
warnings.filterwarnings('ignore')

print("Библиотеки загружены")

 Библиотеки загружены


In [4]:
credits = pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/credits.csv')
keywords = pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/keywords.csv')
links = pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/links.csv')
movies = pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/movies_metadata.csv')
ratings = pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/ratings.csv')

pd.set_option('display.max_columns', None)

def parse_json_column(val):
    if isinstance(val, (list, dict)):
        return val
    if pd.isna(val) or val == '' or val == '[]' or val == '{}':
        return []

    try:
        data = ast.literal_eval(str(val))
        if isinstance(data, list):
            return [item['name'] for item in data if isinstance(item, dict) and 'name' in item]
        if isinstance(data, dict):
            return data.get('name', '')
        return []
    except (ValueError, SyntaxError):
        return []

json_cols_movies = ['genres', 'production_companies', 'production_countries', 'spoken_languages', 'belongs_to_collection']
for col in json_cols_movies:
    movies[col] = movies[col].apply(parse_json_column)

credits['cast'] = credits['cast'].apply(parse_json_column)
credits['crew'] = credits['crew'].apply(parse_json_column)

# 3. Обработка keywords
keywords['keywords'] = keywords['keywords'].apply(parse_json_column)

print("Готово!")
display(movies.head(2))
display(credits.head(2))
display(keywords.head(2))


ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])

movies['id_clean'] = pd.to_numeric(movies['id'], errors='coerce')

invalid_mask = (
    movies['id_clean'].isna() |
    movies['vote_average'].isna() |
    movies['vote_count'].isna() |
    movies.duplicated(subset=['id_clean'], keep='first')
)

movies = movies[~invalid_mask].copy()
movies['id'] = movies['id_clean'].astype(int)
movies = movies.drop(columns=['id_clean'])

print(f"Очистка завершена. Чистых строк в movies: {len(movies)}")

movies['homepage'] = movies['homepage'].notna().astype(int)

if 'poster_path' in movies.columns:
    movies = movies.drop(columns=['poster_path'])


print("Optimization complete!")
print(f"Columns in movies: {movies.columns.tolist()}")
print(f"Columns in ratings: {ratings.columns.tolist()}")
display(movies[['title', 'homepage']].head())
display(ratings.head())



valid_ids = set(movies['id'])

def clean_auxiliary_table(df, name, id_col='id'):
    print(f"--- Cleaning {name} ---")
    initial_count = len(df)

    df_clean = df.drop_duplicates(subset=[id_col], keep='first')
    duplicates_removed = initial_count - len(df_clean)

    df_clean = df_clean[df_clean[id_col].astype(float).astype(int).isin(valid_ids)]
    ids_not_in_movies = (initial_count - duplicates_removed) - len(df_clean)

    print(f"Initial rows: {initial_count}")
    print(f"Duplicates removed: {duplicates_removed}")
    print(f"Rows with IDs not in movies removed: {ids_not_in_movies}")
    print(f"Final clean rows: {len(df_clean)}")
    print("\n")
    return df_clean

credits = clean_auxiliary_table(credits, 'credits')
keywords = clean_auxiliary_table(keywords, 'keywords')

links = clean_auxiliary_table(links.dropna(subset=['tmdbId']), 'links', id_col='tmdbId')

links = links.rename(columns={'tmdbId': 'id'})
links['id'] = links['id'].astype(int)

print("Sync complete! All tables now match the movie metadata subset.")

ratings_mapped = ratings.merge(links[['movieId', 'id']], on='movieId', how='inner')


ratings_mapped = ratings_mapped.rename(columns={'id': 'tmdbId'})
ratings_mapped = ratings_mapped.drop(columns=['movieId'])

ratings = ratings_mapped[ratings_mapped['tmdbId'].isin(valid_ids)].copy()

ratings['tmdbId'] = ratings['tmdbId'].astype(int)

print(f" We now have {len(ratings):,} ratings correctly mapped to TMDB IDs.")

display(ratings.head())

movie_counts = ratings['tmdbId'].value_counts()
valid_movie_ids = movie_counts[movie_counts >= 3].index
ratings = ratings[ratings['tmdbId'].isin(valid_movie_ids)]

print(f"Ratings after filtering movies with < 3 ratings: {len(ratings)}")

user_counts = ratings['userId'].value_counts()
valid_user_ids = user_counts[user_counts >= 5].index
ratings = ratings[ratings['userId'].isin(valid_user_ids)]

print(f"Ratings after filtering users with < 5 ratings: {len(ratings)}")

movie_titles = movies[['id', 'title']]
ratings = pd.merge(ratings, movie_titles, left_on='tmdbId', right_on='id', how='inner')
ratings = ratings.drop(columns=['id'])

print("Ratings DataFrame with movie titles merged:")
display(ratings.head())

Готово!


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,Toy Story Collection,30000000,"[Animation, Comedy, Family]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[Pixar Animation Studios],[United States of America],1995-10-30,373554033.0,81.0,[English],Released,NaN,Toy Story,False,7.7,5415.0
1,False,[],65000000,"[Adventure, Fantasy, Family]",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[TriStar Pictures, Teitler Film, Interscope Co...",[United States of America],1995-12-15,262797249.0,104.0,"[English, Français]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


,cast,crew,id
0,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...","[John Lasseter, Joss Whedon, Andrew Stanton, J...",862
1,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...","[Larry J. Franco, Jonathan Hensleigh, James Ho...",8844


,id,keywords
0,862,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,"[board game, disappearance, based on children'..."


Очистка завершена. Чистых строк в movies: 45430
Optimization complete!
Columns in movies: ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']
Columns in ratings: ['userId', 'movieId', 'rating', 'timestamp']


...

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


--- Cleaning credits ---
Initial rows: 45476
Duplicates removed: 44
Rows with IDs not in movies removed: 3
Final clean rows: 45429


--- Cleaning keywords ---
Initial rows: 46419
Duplicates removed: 987
Rows with IDs not in movies removed: 3
Final clean rows: 45429


--- Cleaning links ---
Initial rows: 45624
Duplicates removed: 30
Rows with IDs not in movies removed: 164
Final clean rows: 45430


Sync complete! All tables now match the movie metadata subset.
 We now have 25,980,582 ratings correctly mapped to TMDB IDs.


,userId,rating,timestamp,tmdbId
0,1,1.0,1425941529,197
1,1,4.5,1425942435,10474
2,1,5.0,1425941523,238
3,1,5.0,1425941546,240
4,1,5.0,1425941556,207


Ratings after filtering movies with < 3 ratings: 25963204
Ratings after filtering users with < 5 ratings: 25929633
Ratings DataFrame with movie titles merged:


,userId,rating,timestamp,tmdbId,title
0,1,1.0,1425941529,197,Braveheart
1,1,4.5,1425942435,10474,The Basketball Diaries
2,1,5.0,1425941523,238,The Godfather
3,1,5.0,1425941546,240,The Godfather: Part II
4,1,5.0,1425941556,207,Dead Poets Society


In [5]:
# Пути к датасетам на kaggle
DATA_PATH = "/kaggle/input/datasets/olgasheh"

# Пути к тестовым файлам
test_files = {
    'very_small': f'{DATA_PATH}/testcsv/test_very_small.csv',
    'small': f'{DATA_PATH}/testcsv/test_small.csv',
    'big': f'{DATA_PATH}/testcsv/test_big.csv'
}

for name, path in test_files.items():
    print(f"{name}: {path}")

very_small: /kaggle/input/datasets/olgasheh/testcsv/test_very_small.csv
small: /kaggle/input/datasets/olgasheh/testcsv/test_small.csv
big: /kaggle/input/datasets/olgasheh/testcsv/test_big.csv


In [6]:
def build_sparse_matrix(ratings_df):
    """Строит CSR матрицу из DataFrame с оценками"""
    users = ratings_df['userId'].astype('int32')
    items = ratings_df['tmdbId'].astype('int32')
    ratings_values = ratings_df['rating'].astype('float32')
    
    # Создаем mapping
    unique_users = users.unique()
    unique_items = items.unique()
    
    user_map = {uid: i for i, uid in enumerate(unique_users)}
    item_map = {iid: i for i, iid in enumerate(unique_items)}
    
    user_idx = users.map(user_map).values
    item_idx = items.map(item_map).values
    
    # COO матрица
    coo = coo_matrix((ratings_values, (user_idx, item_idx)), 
                      shape=(len(unique_users), len(unique_items)))
    
    return coo.tocsr(), user_map, item_map

print(" Функция build_sparse_matrix готова")

 Функция build_sparse_matrix готова


In [7]:
def train_lightfm_model(train_interactions, num_components=50, epochs=30, 
                        learning_rate=0.05, loss='warp', num_threads=4):
    """
    Обучает LightFM модель
    loss='warp' - оптимизирует Precision@k
    """
    model = LightFM(
        no_components=num_components,  # Размер эмбеддингов
        loss=loss,
        learning_rate=learning_rate,
        max_sampled=20,
        item_alpha=1e-6,               # L2 регуляризация для фильмов
        user_alpha=1e-6,               # L2 регуляризация для пользователей
        random_state=42
    )
    
    print(f"Обучение модели с параметрами:")
    print(f"  - no_components: {num_components}")
    print(f"  - epochs: {epochs}")
    print(f"  - learning_rate: {learning_rate}")
    print(f"  - loss: {loss}")
    
    # Обучение
    model.fit(
        train_interactions,
        epochs=epochs,
        num_threads=num_threads,
        verbose=True
    )
    
    return model

print(" Функция train_lightfm_model готова")

 Функция train_lightfm_model готова


In [8]:
def get_lightfm_recommendations(model, user_id, user_map, item_map, 
                                 known_items_set, n_recommendations=50):
    """
    Получает рекомендации для пользователя через LightFM
    """
    if user_id not in user_map:
        return []
    
    user_idx = user_map[user_id]
    n_items = len(item_map)
    
    # Получаем все предсказания для пользователя
    scores = model.predict(user_idx, np.arange(n_items), num_threads=4)
    
    # Сортируем по убыванию
    sorted_items = np.argsort(-scores)
    
    # Фильтруем уже просмотренные
    recommendations = []
    reverse_item_map = {v: k for k, v in item_map.items()}
    
    for item_idx in sorted_items:
        tmdb_id = reverse_item_map[item_idx]
        if tmdb_id not in known_items_set:
            recommendations.append(tmdb_id)
        if len(recommendations) >= n_recommendations:
            break
    
    return recommendations

print(" Функция get_lightfm_recommendations готова")

 Функция get_lightfm_recommendations готова


In [9]:
def precision_at_k(recommended, actual, k):
    recommended_set = set(recommended[:k])
    actual_set = set(actual)
    if not recommended_set:
        return 0.0
    return len(recommended_set.intersection(actual_set)) / k

def recall_at_k(recommended, actual, k):
    recommended_set = set(recommended[:k])
    actual_set = set(actual)
    if not actual_set:
        return 0.0
    return len(recommended_set.intersection(actual_set)) / len(actual_set)

def calculate_dcg(relevance_scores, k):
    dcg = 0.0
    for i in range(min(k, len(relevance_scores))):
        dcg += relevance_scores[i] / np.log2(i + 2)
    return dcg

def ndcg_at_k(recommended, actual, k):
    relevance = [1 if item in actual else 0 for item in recommended]
    dcg = calculate_dcg(relevance, k)
    
    ideal_relevance = [1] * min(len(actual), k)
    idcg = calculate_dcg(ideal_relevance, k)
    
    return dcg / idcg if idcg > 0 else 0.0

def serendipity_at_k(recommended, actual, popularity_dict, k):
    """Serendipity = среднее relevance * (1 - normalized_popularity)"""
    ser_score = 0.0
    actual_set = set(actual)
    
    for item in recommended[:k]:
        if item in actual_set:
            popularity = popularity_dict.get(item, 0.0) / 5.0  # Нормализация
            ser_score += (1 - popularity)
    
    return ser_score / k if k > 0 else 0.0

print(" Все метрики определены")

 Все метрики определены


In [10]:
def evaluate_lightfm_model(ratings_df, test_file_path, dataset_label, 
                           item_popularity_dict, num_components=50, epochs=30):
    """
    Оценка LightFM модели на тестовых данных
    """
    print(f"\n{'='*60}")
    print(f"Оценка LightFM на {dataset_label} датасете")
    print(f"{'='*60}\n")
    
    # 1. Загружаем глобальный тестовый файл
    global_test = pd.read_csv(test_file_path)
    global_test['tmdbId'] = global_test['tmdbId'].astype(int)
    global_test['userId'] = global_test['userId'].astype(int)
    
    print(f"Глобальный тест: {len(global_test):,} рейтингов")
    
    # 2. Отбираем пользователей с >=3 рейтингами в тесте
    test_user_counts = global_test['userId'].value_counts()
    eligible_users = test_user_counts[test_user_counts >= 3].index.tolist()
    print(f"Пользователей с >=3 рейтингами в тесте: {len(eligible_users)}")
    
    # 3. Строим полную матрицу взаимодействий (только для тренировки модели)
    full_interactions, user_map, item_map = build_sparse_matrix(ratings_df)
    print(f"Матрица взаимодействий: {full_interactions.shape}")
    
    # 4. Обучаем модель на всех данных
    model = train_lightfm_model(full_interactions, num_components=num_components, 
                                 epochs=epochs)
    
    # 5. Параметры фильтрации
    MIN_RELEVANT_IN_TEST = 3
    MIN_TRAIN_RATINGS = 10
    
    K_VALUES = [5, 10, 15]
    
    # Хранилище метрик
    metrics = {k: {'precision': [], 'recall': [], 'ndcg': [], 'serendipity': []} 
               for k in K_VALUES}
    
    processed_users = 0
    
    # 6. Проходим по каждому пользователю
    for user_id in tqdm(eligible_users, desc=f"Оценка {dataset_label}"):
        # Полная история пользователя
        user_all_history = ratings_df[ratings_df['userId'] == user_id].sort_values('timestamp')
        
        if len(user_all_history) < MIN_TRAIN_RATINGS + MIN_RELEVANT_IN_TEST:
            continue
        
        # 80/20 хронологический сплит
        split_point = int(len(user_all_history) * 0.8)
        user_train = user_all_history.iloc[:split_point]
        user_test = user_all_history.iloc[split_point:]
        
        if user_train.empty:
            continue
        
        # Известные фильмы (для фильтрации рекомендаций)
        known_movies = set(user_train['tmdbId'])
        user_avg_rating = user_train['rating'].mean()
        
        # Релевантные фильмы в тесте (оценка >=4 и выше средней)
        relevant_items = user_test[
            (user_test['rating'] >= 4) & 
            (user_test['rating'] >= user_avg_rating)
        ]['tmdbId'].tolist()
        
        if len(relevant_items) < MIN_RELEVANT_IN_TEST:
            continue
        
        processed_users += 1
        
        # Получаем рекомендации
        recs = get_lightfm_recommendations(
            model, user_id, user_map, item_map, known_movies, 
            n_recommendations=max(K_VALUES)
        )
        
        if not recs:
            continue
        
        # Считаем метрики для каждого k
        for k in K_VALUES:
            metrics[k]['precision'].append(precision_at_k(recs, relevant_items, k))
            metrics[k]['recall'].append(recall_at_k(recs, relevant_items, k))
            metrics[k]['ndcg'].append(ndcg_at_k(recs, relevant_items, k))
            metrics[k]['serendipity'].append(
                serendipity_at_k(recs, relevant_items, item_popularity_dict, k)
            )
    
    print(f"\n Обработано пользователей: {processed_users}")
    
    # Вывод результатов
    print(f"\n РЕЗУЛЬТАТЫ LightFM на {dataset_label}:\n")
    results_row = {'Model': f'LightFM_{dataset_label}'}
    
    for k in K_VALUES:
        avg_prec = np.mean(metrics[k]['precision']) if metrics[k]['precision'] else 0
        avg_rec = np.mean(metrics[k]['recall']) if metrics[k]['recall'] else 0
        avg_ndcg = np.mean(metrics[k]['ndcg']) if metrics[k]['ndcg'] else 0
        avg_ser = np.mean(metrics[k]['serendipity']) if metrics[k]['serendipity'] else 0
        
        print(f"@k={k}:")
        print(f"  Precision: {avg_prec:.4f}")
        print(f"  Recall:    {avg_rec:.4f}")
        print(f"  NDCG:      {avg_ndcg:.4f}")
        print(f"  Serendipity: {avg_ser:.4f}")
        print()
        
        results_row[f'Precision_{k}'] = avg_prec
        results_row[f'Recall_{k}'] = avg_rec
        results_row[f'nDCG_{k}'] = avg_ndcg
        results_row[f'Serendipity_{k}'] = avg_ser
    
    return pd.DataFrame([results_row]).set_index('Model')

print(" Функция evaluate_lightfm_model готова")

 Функция evaluate_lightfm_model готова


In [11]:
# Создаем словарь популярности фильмов (средняя оценка)
item_popularity = ratings.groupby('tmdbId')['rating'].mean().to_dict()
print(f"Словарь популярности создан для {len(item_popularity)} фильмов")

Словарь популярности создан для 32237 фильмов


In [12]:
# Запуск на big датасете

print("\n" + "="*60)
print("ЗАПУСК НА TEST_BIG")
print("="*60)

# Снижаем параметры для ускорения на big
results_big = evaluate_lightfm_model(
    ratings_df=ratings,
    test_file_path=test_files['big'],
    dataset_label='big',
    item_popularity_dict=item_popularity,
    num_components=60,
    epochs=30
)

display(results_big)


ЗАПУСК НА TEST_BIG

Оценка LightFM на big датасете

Глобальный тест: 518,593 рейтингов
Пользователей с >=3 рейтингами в тесте: 6151
Матрица взаимодействий: (256044, 32237)
Обучение модели с параметрами:
  - no_components: 60
  - epochs: 30
  - learning_rate: 0.05
  - loss: warp


Epoch: 100%|██████████| 30/30 [20:27<00:00, 40.92s/it]


Оценка big:   0%|          | 0/6151 [00:00<?, ?it/s]


 Обработано пользователей: 5009

 РЕЗУЛЬТАТЫ LightFM на big:

@k=5:
  Precision: 0.2150
  Recall:    0.0537
  NDCG:      0.2278
  Serendipity: 0.0479

@k=10:
  Precision: 0.1894
  Recall:    0.0951
  NDCG:      0.2160
  Serendipity: 0.0430

@k=15:
  Precision: 0.1732
  Recall:    0.1299
  NDCG:      0.2133
  Serendipity: 0.0399



,Precision_5,Recall_5,nDCG_5,Serendipity_5,Precision_10,Recall_10,nDCG_10,Serendipity_10,Precision_15,Recall_15,nDCG_15,Serendipity_15
Model,,,,,,,,,,,,
LightFM_big,0.215013,0.053738,0.227789,0.047902,0.189359,0.095055,0.216039,0.042999,0.173195,0.129868,0.213274,0.039946
